# Configuração inicial — executar no navegador do Colab
Autorize este notebook a acessar o segredo `GITHUB_TOKEN` do Colab. Use um token fine-grained limitado a `vicemnovo/TCC_Final`, com permissão **Contents: Read-only**. Execute uma vez; guarde a senha escolhida fora do projeto.

In [ ]:
%pip -q install cryptography
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
root = Path('/content/drive/MyDrive/TCC_Final')
(root / 'credenciais').mkdir(parents=True, exist_ok=True)
target = root / 'credenciais/github.enc'
if target.exists():
    raise FileExistsError(f'Credencial já existe e foi preservada: {target}')

In [ ]:
from getpass import getpass
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from base64 import b64encode, urlsafe_b64encode
import json, os
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Configure o segredo GITHUB_TOKEN no painel do Colab')
password = getpass('Crie uma senha forte para desbloquear no VS Code: ')
confirmation = getpass('Repita a senha: ')
if password != confirmation or len(password) < 12:
    raise ValueError('As senhas devem coincidir e ter pelo menos 12 caracteres')
salt = os.urandom(16)
iterations = 600000
kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=iterations)
ciphertext = Fernet(urlsafe_b64encode(kdf.derive(password.encode()))).encrypt(token.encode())
payload = {'version': 1, 'salt': b64encode(salt).decode(), 'iterations': iterations, 'ciphertext': ciphertext.decode()}
with target.open('x', encoding='utf-8') as f:
    json.dump(payload, f)
os.chmod(target, 0o600)
del token, password, confirmation, ciphertext
print('Credencial criptografada criada no Drive.')